# Semana 35 · ¿Cómo nace, vive y muere una estrella?

Este cuaderno acompaña a la sesión de proyecto de la semana 35: un diagrama de Hertzsprung-Russell con datos reales, líneas de radio constante, la relación masa-luminosidad y la simulación del diagrama de un cúmulo que envejece.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. Cargar los datos

Usamos un subconjunto de la base de datos **HYG** (versión 4.1, de David Nash, licencia CC BY-SA 4.0), que combina los catálogos Hipparcos, Yale y Gliese: todas las estrellas con índice de color a menos de 25 pársecs y todas las más brillantes que la magnitud 4, unas 2800. El archivo está en la carpeta `cuadernos/datos` del repositorio, con una explicación de cada columna.

La celda busca el archivo en tu ordenador y, si no lo encuentra (por ejemplo, en Colab), lo descarga del repositorio. Para eso usa **pandas**, la biblioteca de tablas de datos más usada en Python.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

URL = "https://raw.githubusercontent.com/alabcar/hola-universo/main/cuadernos/datos/estrellas_hyg.csv"
locales = ["cuadernos/datos/estrellas_hyg.csv", "../datos/estrellas_hyg.csv", "datos/estrellas_hyg.csv"]
ruta = next((r for r in locales if os.path.exists(r)), URL)
estrellas = pd.read_csv(ruta)
print(f"{len(estrellas)} estrellas leídas de {ruta}")
estrellas.head(10)

Cada fila es una estrella; cada columna, una propiedad. `dist_pc` es la distancia en pársecs, `mag` la magnitud aparente, `absmag` la absoluta, `b_v` el índice de color y `espectro` el tipo espectral.

## 2. El diagrama HR de los observadores

Magnitud absoluta frente a índice de color. Las magnitudes van al revés, así que invertimos el eje vertical.

In [ ]:
famosas = {"Sol": "Sol", "Sirius": "Sirio", "Procyon": "Proción", "Vega": "Vega", "Arcturus": "Arturo",
           "Capella": "Capella", "Aldebaran": "Aldebarán", "Spica": "Espiga", "Betelgeuse": "Betelgeuse",
           "Antares": "Antares", "Rigel": "Rigel", "Deneb": "Deneb", "Proxima Centauri": "Próxima Centauri",
           "Van Maanen's Star": "Estrella de van Maanen"}
sirio_b = estrellas[(estrellas.constelacion == "CMa") & (estrellas.dist_pc < 2.7) & estrellas.espectro.str.startswith("DA", na=False)]

fig, ax = plt.subplots(figsize=(9, 10))
ax.scatter(estrellas.b_v, estrellas.absmag, s=4, c=estrellas.b_v, cmap="RdYlBu_r", vmin=-0.3, vmax=2.0)
for nombre_hyg, nombre in famosas.items():
    fila = estrellas[estrellas.nombre == nombre_hyg]
    if len(fila):
        ax.annotate(nombre, (fila.b_v.iloc[0], fila.absmag.iloc[0]), fontsize=8, xytext=(5, 3), textcoords="offset points")
        ax.plot(fila.b_v, fila.absmag, "ko", ms=4)
if len(sirio_b):
    ax.plot(sirio_b.b_v, sirio_b.absmag, "ko", ms=4)
    ax.annotate("Sirio B", (sirio_b.b_v.iloc[0], sirio_b.absmag.iloc[0]), fontsize=8, xytext=(5, 3), textcoords="offset points")
ax.invert_yaxis(); ax.set_xlim(-0.5, 2.3)
ax.set_xlabel("índice de color B − V (azul ← → rojo)"); ax.set_ylabel("magnitud absoluta M")
ax.set_title("Diagrama de Hertzsprung-Russell (datos HYG)"); ax.grid(alpha=0.3)
plt.show()

Distingue las tres zonas: la **secuencia principal**, la banda diagonal; las **gigantes**, el grupo arriba a la derecha; y las **enanas blancas**, el grupo disperso abajo a la izquierda.

Cuidado con un sesgo: las estrellas de menos de 25 pc son una muestra justa del vecindario, pero las más brillantes que la magnitud 4 están sesgadas hacia las muy luminosas, que se ven desde muy lejos. Lo comprobamos separando las dos muestras:

In [ ]:
cercanas = estrellas[estrellas.dist_pc < 25]
brillantes = estrellas[estrellas.mag < 4]
fig, axs = plt.subplots(1, 2, figsize=(12, 6), sharey=True)
for ax, datos, titulo in [(axs[0], cercanas, f"a menos de 25 pc ({len(cercanas)})"),
                          (axs[1], brillantes, f"más brillantes que m = 4 ({len(brillantes)})")]:
    ax.scatter(datos.b_v, datos.absmag, s=4, c=datos.b_v, cmap="RdYlBu_r", vmin=-0.3, vmax=2.0)
    ax.set_title(titulo); ax.set_xlabel("B − V"); ax.grid(alpha=0.3); ax.set_xlim(-0.5, 2.3)
axs[0].invert_yaxis(); axs[0].set_ylabel("magnitud absoluta M")
plt.show()
mas_debiles_que_sol = (cercanas.absmag > 4.85).mean()
print(f"De las estrellas cercanas, el {100 * mas_debiles_que_sol:.0f} % son menos luminosas que el Sol.")

La mayoría de las estrellas del vecindario son enanas rojas, más débiles que el Sol, y ninguna se ve a simple vista. Las estrellas que vemos en el cielo no son las típicas: son las más luminosas.

## 3. El diagrama de los teóricos y las líneas de radio constante

Pasamos el índice de color a temperatura con la fórmula de Ballesteros (semana 33) y la magnitud absoluta a luminosidad. Usamos la luminosidad **visual**, sin corregir la luz fuera del visible: para las estrellas muy calientes o muy frías, la luminosidad total es bastante mayor. Las líneas de radio constante salen de $L = R^2 (T/T_\odot)^4$ en unidades solares.

In [ ]:
def temperatura(bv):
    return 4600 * (1 / (0.92 * bv + 1.7) + 1 / (0.92 * bv + 0.62))

T = temperatura(estrellas.b_v.clip(-0.3, 2.0))
L = 10 ** (-0.4 * (estrellas.absmag - 4.83))

fig, ax = plt.subplots(figsize=(9, 9))
ax.scatter(T, L, s=4, c=estrellas.b_v, cmap="RdYlBu_r", vmin=-0.3, vmax=2.0)
T_linea = np.logspace(np.log10(2500), np.log10(30000), 100)
for R in [0.01, 0.1, 1, 10, 100, 1000]:
    ax.plot(T_linea, R**2 * (T_linea / 5772) ** 4, "k--", lw=0.6)
    ax.text(3000, R**2 * (3000 / 5772) ** 4, f"{R} R☉", fontsize=8)
ax.set_xscale("log"); ax.set_yscale("log"); ax.invert_xaxis()
ax.set_xlim(30000, 2500); ax.set_ylim(1e-5, 1e6)
ax.set_xlabel("temperatura (K)"); ax.set_ylabel("luminosidad visual (L☉)"); ax.grid(alpha=0.3, which="both")
plt.show()

## 4. La relación masa-luminosidad

Masas y luminosidades totales **aproximadas** de algunas estrellas de la secuencia principal cuyas masas se conocen bien (casi todas, por estar en sistemas dobles). Son valores redondeados de la literatura, suficientes para ver la tendencia.

In [ ]:
ml = pd.DataFrame([
    ("Próxima Centauri", 0.12, 0.0016), ("Estrella de Barnard", 0.16, 0.0035), ("61 Cygni B", 0.63, 0.085),
    ("61 Cygni A", 0.70, 0.15), ("Alfa Centauri B", 0.91, 0.50), ("Sol", 1.00, 1.00),
    ("Alfa Centauri A", 1.10, 1.52), ("Proción A", 1.50, 6.9), ("Altair", 1.8, 10.6),
    ("Sirio A", 2.06, 25.4), ("Vega", 2.1, 40), ("Regulus", 3.8, 290), ("Espiga A", 11.4, 20500),
], columns=["estrella", "masa", "luminosidad"])

pendiente, ordenada = np.polyfit(np.log10(ml.masa), np.log10(ml.luminosidad), 1)
print(f"L ≈ {10**ordenada:.2f} · M^{pendiente:.2f}")

fig, ax = plt.subplots(figsize=(7, 6))
ax.loglog(ml.masa, ml.luminosidad, "o")
for _, f in ml.iterrows():
    ax.annotate(f.estrella, (f.masa, f.luminosidad), fontsize=7, xytext=(4, -8), textcoords="offset points")
m = np.logspace(-1, 1.2, 50)
ax.loglog(m, 10**ordenada * m**pendiente, "r-", label=f"ajuste: pendiente {pendiente:.2f}")
ax.loglog(m, m**3.5, "k--", lw=0.8, label="M^3,5")
ax.set_xlabel("masa (M☉)"); ax.set_ylabel("luminosidad (L☉)"); ax.legend(); ax.grid(alpha=0.3, which="both")
plt.show()

La pendiente sale cerca de 4: el exponente de 3,5 es una media, y en realidad cambia según el rango de masas (más bajo para las enanas rojas y las estrellas muy masivas).

## 5. Un cúmulo que envejece (simulación)

Ahora un modelo de juguete: un cúmulo de 3000 estrellas que nacen a la vez, con masas repartidas como en los cúmulos reales (muchas pequeñas y pocas grandes, según la **función de masa inicial** de Salpeter, $dN/dM \propto M^{-2{,}35}$). Cada estrella vive en la secuencia principal $t = 10^{10}\,M^{-2{,}5}$ años, con $L = M^{3{,}5}$ y una temperatura que crece con la masa. Al acabar su vida en la secuencia, pasa un 10 % más de tiempo como gigante roja (la colocamos a unos 4500 K y 50 veces más luminosa) y luego desaparece del diagrama. **Todo es simplificado**: sirve para ver la idea del punto de desvío, no para medir edades.

In [ ]:
rng = np.random.default_rng(35)
n, m_min, m_max, alfa = 3000, 0.3, 20, 2.35
u = rng.random(n)                       # muestreo de una ley de potencias por inversión
masas = (m_min**(1 - alfa) + u * (m_max**(1 - alfa) - m_min**(1 - alfa))) ** (1 / (1 - alfa))
vida = 1e10 * masas ** -2.5
L_ms = masas ** 3.5 * 10 ** rng.normal(0, 0.08, n)        # un poco de dispersión, como en un cúmulo real
T_ms = 5772 * masas ** 0.5 * 10 ** rng.normal(0, 0.01, n)

fig, axs = plt.subplots(1, 4, figsize=(16, 5), sharex=True, sharey=True)
for ax, edad in zip(axs, [1e7, 1e8, 1e9, 5e9]):
    en_ms = vida > edad
    gigante = (vida <= edad) & (edad < 1.1 * vida)
    ax.scatter(T_ms[en_ms], L_ms[en_ms], s=3, c="tab:blue")
    ax.scatter(np.full(gigante.sum(), 4500) * rng.normal(1, 0.05, gigante.sum()), 50 * L_ms[gigante], s=8, c="tab:red")
    desvio = masas[en_ms].max()
    ax.set_title(f"{edad / 1e6:,.0f} millones de años\ndesvío en {desvio:.1f} M☉".replace(",", " "))
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("T (K)"); ax.grid(alpha=0.3, which="both")
axs[0].set_xlim(30000, 2500); axs[0].set_ylabel("L (L☉)")
plt.tight_layout(); plt.show()

Con la edad, la secuencia principal se va «comiendo» desde arriba: el punto de desvío baja a masas cada vez menores. Comparando el diagrama de un cúmulo real con modelos detallados de este tipo (llamados **isócronas**, «igual tiempo»), los astrónomos miden su edad.

## 6. Experimenta

1. En el diagrama HR real, cuenta cuántas estrellas cercanas hay con $M > 10$ y $B - V < 0{,}6$ (enanas blancas). ¿Qué fracción del total son?
2. Busca en la tabla las estrellas de tipo espectral M (la columna `espectro` empieza por M) y dibuja solo esas. ¿Dónde caen?
3. En la simulación del cúmulo, calcula la edad para la que el punto de desvío está en 2,5 masas solares y compárala con el resultado del ejercicio de la sesión 2.

In [ ]:
# Tu experimento aquí